# 第 26 章：Gaia HR 图与恒星分类案例

这个 notebook 演示一个更接近真实项目的小案例：

- 从 Gaia 风格表格开始做数据质量筛选
- 用视差和表观星等计算绝对星等
- 在 HR 图空间里做 k-means 聚类
- 用最近中心分类器做一个最小监督分类流程


In [ ]:
from __future__ import annotations

import csv
import math
import os
from pathlib import Path

DATA_PATH = Path("../../data/small/gaia_hr_case_demo.csv").resolve()

with DATA_PATH.open(newline="", encoding="utf-8") as handle:
    rows = []
    for raw in csv.DictReader(handle):
        rows.append({
            "source_id": raw["source_id"],
            "bp_rp": float(raw["bp_rp"]),
            "parallax_mas": float(raw["parallax_mas"]),
            "parallax_error_mas": float(raw["parallax_error_mas"]),
            "g_mag": float(raw["g_mag"]),
            "stellar_class": raw["stellar_class"],
        })

print(f"Loaded {len(rows)} Gaia-style samples from {DATA_PATH.name}")
class_counts = {}
for row in rows:
    class_counts[row["stellar_class"]] = class_counts.get(row["stellar_class"], 0) + 1
print("class counts:", class_counts)
print("first row:", rows[0])


In [ ]:
for row in rows:
    row["parallax_over_error"] = row["parallax_mas"] / row["parallax_error_mas"]

clean_rows = [row for row in rows if row["parallax_over_error"] > 5.0]
filtered_out_rows = [row for row in rows if row["parallax_over_error"] <= 5.0]

print("clean sample size:", len(clean_rows))
print("filtered out ids:", [row["source_id"] for row in filtered_out_rows])
print("parallax_over_error range in clean data:", (
    min(row["parallax_over_error"] for row in clean_rows),
    max(row["parallax_over_error"] for row in clean_rows),
))


In [ ]:
for row in clean_rows:
    row["absolute_g_mag"] = row["g_mag"] + 5.0 * math.log10(row["parallax_mas"]) - 10.0

print("Example absolute magnitudes:")
for row in clean_rows[:5]:
    print(row["source_id"], row["stellar_class"], round(row["absolute_g_mag"], 2))

class_ranges = {}
for row in clean_rows:
    class_ranges.setdefault(row["stellar_class"], {"bp_rp": [], "absolute_g_mag": []})
    class_ranges[row["stellar_class"]]["bp_rp"].append(row["bp_rp"])
    class_ranges[row["stellar_class"]]["absolute_g_mag"].append(row["absolute_g_mag"])
print("HR regions by class:")
for stellar_class, values in class_ranges.items():
    print(
        stellar_class,
        "bp_rp=", (round(min(values["bp_rp"]), 2), round(max(values["bp_rp"]), 2)),
        "M_G=", (round(min(values["absolute_g_mag"]), 2), round(max(values["absolute_g_mag"]), 2)),
    )


In [ ]:
feature_names = ["bp_rp", "absolute_g_mag"]
feature_means = {}
feature_stds = {}
for feature_name in feature_names:
    values = [row[feature_name] for row in clean_rows]
    mean_value = sum(values) / len(values)
    variance = sum((value - mean_value) ** 2 for value in values) / len(values)
    feature_means[feature_name] = mean_value
    feature_stds[feature_name] = math.sqrt(variance) or 1.0

for row in clean_rows:
    row["hr_vector"] = [
        (row[feature_name] - feature_means[feature_name]) / feature_stds[feature_name]
        for feature_name in feature_names
    ]

print("standardized feature means:", {name: round(value, 3) for name, value in feature_means.items()})
print("standardized feature stds:", {name: round(value, 3) for name, value in feature_stds.items()})


In [ ]:
def squared_distance(left_vector, right_vector):
    return sum((left - right) ** 2 for left, right in zip(left_vector, right_vector))


seed_ids = ["WD02", "RG03", "MS04"]
centers = [next(row["hr_vector"] for row in clean_rows if row["source_id"] == source_id)[:] for source_id in seed_ids]

for _ in range(20):
    groups = [[] for _ in centers]
    for row in clean_rows:
        distances = [squared_distance(row["hr_vector"], center) for center in centers]
        cluster_index = min(range(len(distances)), key=distances.__getitem__)
        groups[cluster_index].append(row)
    new_centers = []
    for group, center in zip(groups, centers):
        if not group:
            new_centers.append(center)
            continue
        new_centers.append([sum(row["hr_vector"][i] for row in group) / len(group) for i in range(2)])
    movement = sum(squared_distance(left, right) for left, right in zip(centers, new_centers))
    centers = new_centers
    if movement < 1e-12:
        break

cluster_assignments = {}
for row in clean_rows:
    distances = [squared_distance(row["hr_vector"], center) for center in centers]
    cluster_assignments[row["source_id"]] = min(range(len(distances)), key=distances.__getitem__)

cluster_summary = {}
for row in clean_rows:
    cluster_index = cluster_assignments[row["source_id"]]
    cluster_summary.setdefault(cluster_index, {})
    cluster_summary[cluster_index][row["stellar_class"]] = cluster_summary[cluster_index].get(row["stellar_class"], 0) + 1
print("k-means seed ids:", seed_ids)
print("cluster composition:", cluster_summary)


In [ ]:
test_ids = {"MS03", "MS06", "RG02", "RG04", "WD03", "WD05"}
train_rows = [row for row in clean_rows if row["source_id"] not in test_ids]
test_rows = [row for row in clean_rows if row["source_id"] in test_ids]

class_centroids = {}
for stellar_class in sorted({row["stellar_class"] for row in train_rows}):
    group_vectors = [row["hr_vector"] for row in train_rows if row["stellar_class"] == stellar_class]
    class_centroids[stellar_class] = [sum(values[i] for values in group_vectors) / len(group_vectors) for i in range(2)]


def predict_class(row):
    distances = {stellar_class: squared_distance(row["hr_vector"], centroid) for stellar_class, centroid in class_centroids.items()}
    return min(distances, key=distances.get)


predictions = []
for row in test_rows:
    predicted = predict_class(row)
    predictions.append((row["source_id"], row["stellar_class"], predicted))
print("nearest-centroid test predictions:")
for item in predictions:
    print(item)
accuracy = sum(true_label == predicted_label for _, true_label, predicted_label in predictions) / len(predictions)
print("test accuracy:", accuracy)


In [ ]:
confusion = {}
for _, true_label, predicted_label in predictions:
    confusion.setdefault(true_label, {})
    confusion[true_label][predicted_label] = confusion[true_label].get(predicted_label, 0) + 1
print("confusion matrix:", confusion)

print("Physical interpretation:")
print("  main_sequence -> redder stars tend to be intrinsically fainter in this toy sample.")
print("  giant -> red but bright objects occupy a distinct upper branch.")
print("  white_dwarf -> blue and faint objects occupy the lower-left region of the HR diagram.")


In [ ]:
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    print("matplotlib 未安装；已跳过 HR 图绘制。")
else:
    class_colors = {"main_sequence": "tab:blue", "giant": "tab:red", "white_dwarf": "tab:green"}
    fig, ax = plt.subplots(figsize=(6, 4))
    for stellar_class, color in class_colors.items():
        group = [row for row in clean_rows if row["stellar_class"] == stellar_class]
        ax.scatter([row["bp_rp"] for row in group], [row["absolute_g_mag"] for row in group], label=stellar_class, color=color)
    ax.set_title("Teaching Gaia HR Diagram")
    ax.set_xlabel("BP-RP [mag]")
    ax.set_ylabel("Absolute G Magnitude [mag]")
    ax.invert_yaxis()
    ax.legend()
    
    if os.environ.get('AIFA_SKIP_PLOTS') == '1':
        plt.close(figure if 'figure' in locals() else fig)
    else:
        plt.show()


In [ ]:
try:
    from sklearn.cluster import KMeans
    from sklearn.neighbors import NearestCentroid
except ModuleNotFoundError:
    print("scikit-learn 未安装；已跳过标准库案例实现。")
else:
    feature_matrix = [row["hr_vector"] for row in clean_rows]
    kmeans = KMeans(n_clusters=3, random_state=0, n_init=10)
    predicted_clusters = kmeans.fit_predict(feature_matrix)
    print("sklearn k-means labels:", predicted_clusters.tolist())
    classifier = NearestCentroid()
    classifier.fit([row["hr_vector"] for row in train_rows], [row["stellar_class"] for row in train_rows])
    sklearn_predictions = classifier.predict([row["hr_vector"] for row in test_rows])
    print("sklearn nearest-centroid predictions:", sklearn_predictions.tolist())


## 导出 Ch26 最小 case package

下面这个单元会把本章结果导出到 `results/ch26_gaia_hr_case/`。它示范 HR 图案例如何把数据质量、结构探索、监督分类和边界解释放进同一个 Model Experiment Packet。

In [ ]:
RESULTS_DIR = Path("results/ch26_gaia_hr_case")
FIGURES_DIR = RESULTS_DIR / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

prediction_rows = [
    {
        "source_id": source_id,
        "true_class": true_label,
        "predicted_class": predicted_label,
        "correct": true_label == predicted_label,
    }
    for source_id, true_label, predicted_label in predictions
]

with (RESULTS_DIR / "confusion_matrix.csv").open("w", newline="", encoding="utf-8") as handle:
    labels = sorted({row["stellar_class"] for row in clean_rows})
    writer = csv.DictWriter(handle, fieldnames=["true_class"] + labels)
    writer.writeheader()
    for true_label in labels:
        row = {"true_class": true_label}
        for predicted_label in labels:
            row[predicted_label] = confusion.get(true_label, {}).get(predicted_label, 0)
        writer.writerow(row)

boundary_rows = []
for row in test_rows:
    distances = sorted(
        (squared_distance(row["hr_vector"], centroid), stellar_class)
        for stellar_class, centroid in class_centroids.items()
    )
    boundary_rows.append({
        "source_id": row["source_id"],
        "stellar_class": row["stellar_class"],
        "predicted_class": predict_class(row),
        "bp_rp": row["bp_rp"],
        "absolute_g_mag": row["absolute_g_mag"],
        "parallax_over_error": row["parallax_over_error"],
        "nearest_distance": math.sqrt(distances[0][0]),
        "runner_up_class": distances[1][1],
        "distance_margin": distances[1][0] - distances[0][0],
    })
boundary_rows.sort(key=lambda row: row["distance_margin"])

with (RESULTS_DIR / "boundary_examples.csv").open("w", newline="", encoding="utf-8") as handle:
    fieldnames = [
        "source_id", "stellar_class", "predicted_class", "bp_rp", "absolute_g_mag",
        "parallax_over_error", "nearest_distance", "runner_up_class", "distance_margin",
    ]
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(boundary_rows)

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    print(f"Figure export skipped: {exc}")
else:
    class_colors = {"main_sequence": "tab:blue", "giant": "tab:red", "white_dwarf": "tab:green"}
    figure, axis = plt.subplots(figsize=(6, 4.6))
    for stellar_class, color in class_colors.items():
        group = [row for row in clean_rows if row["stellar_class"] == stellar_class]
        axis.scatter([row["bp_rp"] for row in group], [row["absolute_g_mag"] for row in group], label=stellar_class, color=color, s=55)
    for row in boundary_rows[:3]:
        axis.text(row["bp_rp"] + 0.02, row["absolute_g_mag"] + 0.05, row["source_id"], fontsize=8)
    axis.set_title("HR diagram with boundary examples")
    axis.set_xlabel("BP-RP [mag]")
    axis.set_ylabel("Absolute G magnitude [mag]")
    axis.invert_yaxis()
    axis.legend()
    figure.tight_layout()
    figure.savefig(FIGURES_DIR / "hr_diagram_boundary_examples.png", dpi=160)
    plt.close(figure)

(RESULTS_DIR / "dataset_contract_gaia_hr.md").write_text(f"""# Dataset Contract: Ch26 Gaia HR

## Task Definition
- Scientific question: Can Gaia-style color and absolute magnitude recover teaching-scale HR diagram structures?
- ML task: clustering and classification
- Prediction target: `stellar_class` for the supervised step
- Intended use: teaching-scale HR structure workflow

## Sample Definition
- Sample unit: one Gaia-style source row
- Sample ID: `source_id`
- Data file: `{DATA_PATH.relative_to(Path.cwd()) if DATA_PATH.is_relative_to(Path.cwd()) else DATA_PATH.name}`

## Input Features
- HR features: `bp_rp`, `absolute_g_mag`
- Derived quantity: `absolute_g_mag = g_mag + 5 log10(parallax_mas) - 10`
- Quality field: `parallax_over_error`

## Target / Label
- Target: `stellar_class`
- Label source: teaching reference class
- Ambiguous cases: HR boundary objects, simplified stellar classes

## Selection / Split
- Quality cut: `parallax_over_error > 5`
- Removed IDs: {', '.join(row['source_id'] for row in filtered_out_rows) or 'none'}
- Train size: {len(train_rows)}
- Test size: {len(test_rows)}
- Test IDs: {', '.join(row['source_id'] for row in test_rows)}

## Known Limits
- No extinction correction
- No binary/variable-star modeling
- Simplified class labels and small teaching sample
""", encoding="utf-8")

(RESULTS_DIR / "model_experiment_record_gaia_hr.md").write_text(f"""# Model Experiment Record: Ch26 Gaia HR

## Task
- Scientific question: Can HR coordinates organize stellar structure and support a simple classifier?
- ML task: clustering and classification
- Prediction target: `stellar_class`

## Dataset Contract
- Dataset Contract link: `dataset_contract_gaia_hr.md`

## Baseline / Model
- Structure exploration: hand-seeded k-means in standardized HR space
- Supervised baseline: nearest-centroid classifier in standardized HR space
- Algorithm Card sidebar links: Ch23 k-means; Ch19 nearest-centroid classification; Ch20 diagnostics

## Evaluation
- Test accuracy: {accuracy:.3f}
- Evaluation artifacts: `confusion_matrix.csv`, `figures/hr_diagram_boundary_examples.png`
- Cluster summary: {cluster_summary}

## Error Analysis
- Error/boundary artifact: `boundary_examples.csv`
- Main check: boundary or low-margin objects in HR space

## Limit
- Supported claim: the teaching sample recovers clear HR structures with simple methods.
- Unsupported claim: the same classifier is ready for full Gaia survey inference.
""", encoding="utf-8")

(RESULTS_DIR / "trust_statement_gaia_hr.md").write_text("""# Trust Statement: Ch26 Gaia HR

## Model Output
- Result: HR-space clustering and nearest-centroid stellar classification
- Main diagnostic: HR diagram, confusion matrix, boundary examples

## Distribution Status
- In-distribution for clean teaching rows after parallax quality cut
- Unclear for low-parallax-SNR, extincted, binary, variable, or rare stellar populations

## Uncertainty
- Main source: simplified labels, parallax quality, missing extinction/binary treatment
- Estimated by: quality cut comparison and boundary-example review

## Interpretability
- Main feature dependence: color and absolute magnitude
- Not causal proof: the classifier reflects HR-space separation in this teaching sample, not a full stellar-evolution model

## Failure Boundary
- Known failure region: class boundaries, low parallax quality, extinction, binaries, variables, survey selection effects
- Human review needed: rows in `boundary_examples.csv`

## Claim Boundary
- Supported claim: physical HR structure should be inspected before modeling.
- Unsupported claim: teaching-sample accuracy generalizes to full Gaia data.
""", encoding="utf-8")

print("Wrote Ch26 case package to", RESULTS_DIR)
print("Expected files:", sorted(path.name for path in RESULTS_DIR.iterdir()))